# US-082 - Dataset Italia COMPLETO + re-evaluacion honesta del TL

### El F1 0.13 NO era el dataset: era el 1 % del dataset

**Equipo 17** - AgroSatCopilot - Transfer learning mediterraneo (EPIC 12)

---

El F1-macro **0.13** del TL Italia (US-079) no se debio a un dataset pobre ni a un bug: el modelo entreno y evaluo sobre el **PILOTO del 1 %** (20 patches / 884 parcelas) en vez del dataset completo (**1,438 patches / 107,493 parcelas**, ya en disco). Esta US re-extrae AlphaEarth sobre el dataset completo y re-evalua el transfer por **tres vias** que Arthur decidio comparar:

- **Via A** - las **39 hojas HCAT nativas** tal cual (sin reagrupar clases).
- **Via B** - las hojas italianas **mapeadas al espacio de entrada del campeon** (el crosswalk conservado a PASTIS-18 / france-12 que los miembros franceses ya conocen).
- **Via C** - el **procedimiento completo replicado** de extremo a extremo sobre el dataset nuevo (extraccion AlphaEarth -> entreno xgb/TSViT/U-TAE -> OOF fold-5 -> Voting-3 -> eval), igual que se hizo con PASTIS-Francia.

> **Solo valores reales.** Toda metrica se lee de los artefactos REALES que produce la corrida de extraccion + entrenamiento en la H100 (el parquet de features completo, el OOF, los report JSON). Si un artefacto no existe aun, la celda lo dice y muestra el estado **pendiente**, nunca un numero inventado. NO se reagrupan clases.

In [ ]:
# Parametros (papermill). Sobreescribe con `papermill -p <name> <value>`.
data_dir = "data\pastis_italia_2018"   # dataset Italia completo (US-078, 1438 patches)
features_full = "data\features\alphaearth_italia_2018_full1438.parquet"   # features AlphaEarth del dataset completo (_full1438)
features_pilot = "data\features\alphaearth_italia_2018.parquet"   # features del piloto del 1 % (para el A/B)
report_dir = "reports\transfer_italia"   # report JSON del runner (metricas reales del TL)
year = 2018   # campaña del embedding anual (Italia 2018)
pastis_dates = 43   # fechas medias PASTIS-Francia (referencia)
pastis_patches = 2468   # patches PASTIS-Francia (referencia)
pastis_parcels = 124000   # parcelas PASTIS-Francia (referencia)

## Preparacion del entorno

Resolvemos la raiz del repositorio y forzamos UTF-8 en la salida (la consola de Windows usa cp1252 y la prosa/logs llevan acentos). Todo lo que sigue lee de rutas del repo; nada se descarga ni se fabrica.

In [ ]:
import sys, io, json
from pathlib import Path

if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

def _find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for parent in [cur, *cur.parents]:
        if (parent / 'pyproject.toml').is_file():
            return parent
    return cur

REPO = _find_repo_root(Path.cwd())
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print('repo root:', REPO)

## 1. Causa raiz: el piloto del 1 %

El runner de features uso 20 patches (884 parcelas) en vez de los 1,438 patches (107,493 parcelas) reales. Verificamos el conteo real del dataset en disco -- es la evidencia que motiva toda la US.

In [ ]:
import polars as pl
meta = pl.read_parquet(Path(REPO) / data_dir / 'metadata.parquet')
n_patches = meta.height
n_parcels = int(meta['n_parcelas'].sum()) if 'n_parcelas' in meta.columns else None
print(f'Dataset completo en disco: {n_patches} patches, {n_parcels} parcelas')
print(f'PASTIS-Francia (referencia): {pastis_patches} patches, {pastis_parcels} parcelas')
print(f'Italia / PASTIS: {100*n_patches/pastis_patches:.0f}% patches, {100*n_parcels/pastis_parcels:.0f}% parcelas')
# Piloto del 1 %: 20 patches / 884 parcelas (lo que uso US-079).
pilot_path = Path(REPO) / features_pilot
if pilot_path.is_file():
    pilot_df = pl.read_parquet(pilot_path)
    print(f'Piloto del 1 %% (US-079): {pilot_df.height} filas de parcela')
else:
    print('PENDIENTE: piloto no presente en disco (no bloquea el dataset completo)')

## 2. EDA formal del dataset completo

`ml.transfer.eda_italia.compute_italia_eda` resume volumen, soporte por clase, el techo temporal (fechas vs PASTIS) y la co-ocurrencia inter-clase, todo desde el `metadata.parquet` real. Las clases con >= 200 patches son candidatas a F1 alto.

In [ ]:
from ml.transfer.eda_italia import compute_italia_eda
eda = compute_italia_eda(Path(REPO) / data_dir)
print(json.dumps(eda.summary(), indent=2))
print(f'\nTecho temporal: media {eda.date_stats["mean"]:.1f} fechas (PASTIS {pastis_dates}); {eda.n_patches_weak_phenology} patches con < 16 fechas')
# Tabla de soporte por clase (las candidatas con >= 200 patches primero).
pl.DataFrame(eda.per_class).head(40)

## 3. Separabilidad post-extraccion (JM / Bhattacharyya)

Con el dataset completo (cientos de parcelas/clase) la covarianza de 64 dims ya es estimable, asi que la distancia Jeffries-Matusita es confiable -- a diferencia del JM=2.0 espurio del piloto (covarianza degenerada con 1-5 parcelas/clase). Requiere el parquet de features completo (`_full1438`); si no existe aun, la celda lo dice.

In [ ]:
feat_path = Path(REPO) / features_full
if feat_path.is_file():
    from ml.transfer.separability_italia import compute_separability
    sep = compute_separability(features_path=feat_path, italia_root=Path(REPO) / data_dir)
    print(json.dumps(sep.summary() if hasattr(sep, 'summary') else {}, indent=2, default=str))
    display(pl.DataFrame(sep.per_class) if hasattr(sep, 'per_class') else sep)
else:
    print(f'PENDIENTE: {features_full} no existe. Corre la extraccion AlphaEarth full '
          '1438 en la H100 y vuelve a ejecutar este notebook con papermill.')

## 4. Las tres vias comparadas

`ml.transfer.eval_three_ways.compare_three_ways` puntua las predicciones densas del Voting-3 Italia en las tres vias: **A** (39 hojas nativas), **B** (mapeo al crosswalk PASTIS) y **C** (procedimiento completo). La tabla muestra macro-F1, nº de clases >= 0.6 y >= 0.8 por via. Requiere las predicciones densas del OOF fold-5 (report del runner); si no existen, estado pendiente honesto.

In [ ]:
rep = Path(REPO) / report_dir / 'three_ways_comparison.json'
if rep.is_file():
    table = json.loads(rep.read_text(encoding='utf-8'))
    display(pl.DataFrame(table['table'] if isinstance(table, dict) else table))
else:
    print(f'PENDIENTE: {rep} no existe. La via C (entreno + OOF + voto) corre en la '
          'H100; este notebook puebla la tabla al re-ejecutarse con el report presente.')

## 5. A/B: piloto (884) vs completo (107k)

La tabla que prueba que el muestreo era la causa: el mismo pipeline sobre el piloto del 1 % vs el dataset completo. Si el F1 por-clase sube al pasar de 884 a 107k parcelas, la causa raiz queda demostrada (no el domain-shift).

In [ ]:
ab = Path(REPO) / report_dir / 'ab_pilot_vs_full.json'
if ab.is_file():
    display(pl.DataFrame(json.loads(ab.read_text(encoding='utf-8'))))
else:
    print(f'PENDIENTE: {ab} no existe. Se genera tras re-entrenar sobre el dataset '
          'completo (H100) y contrastar con el OOF del piloto.')

## 6. F1 estratificado por nº de fechas (techo temporal)

Italia tiene media 24.3 fechas (vs 43 PASTIS) y muy variable (9-40). Estratificar el F1 por nº de fechas del patch aisla el techo temporal honesto: los patches de pocas fechas padean fuerte a 32 timesteps y su fenologia es pobre.

In [ ]:
strat = Path(REPO) / report_dir / 'f1_by_date_count.json'
if strat.is_file():
    display(pl.DataFrame(json.loads(strat.read_text(encoding='utf-8'))))
else:
    print(f'PENDIENTE: {strat} no existe. Se genera en la eval del TL completo (H100).')

## 7. Veredicto: cuantas clases rescatan de verdad (KPI-2)

El objetivo honesto NO es un solo F1-macro-39 global (cuyo techo lo fijan las ~23 clases de cola minoritarias reales), sino **cuantas clases superan F1 >= 0.6 y >= 0.8** con el dataset completo. Target: >= 10 clases >= 0.6 y >= 5 clases >= 0.8.

In [ ]:
rep = Path(REPO) / report_dir / 'three_ways_comparison.json'
if rep.is_file():
    table = json.loads(rep.read_text(encoding='utf-8'))
    rows = table['table'] if isinstance(table, dict) else table
    for r in rows:
        print(f"Via {r['via']} ({r['label_space']}): macro-F1 {r['macro_f1']}, "
              f"{r['n_classes_ge_0.6']} clases >= 0.6, {r['n_classes_ge_0.8']} >= 0.8")
    via_a = next((r for r in rows if r['via'] == 'A'), None)
    if via_a:
        ok6 = via_a['n_classes_ge_0.6'] >= 10
        ok8 = via_a['n_classes_ge_0.8'] >= 5
        print(f'\nKPI-2 (via A nativa): >=10 clases>=0.6 {"OK" if ok6 else "NO"}; '
              f'>=5 clases>=0.8 {"OK" if ok8 else "NO"}')
else:
    print('PENDIENTE: sin el report de las tres vias no hay veredicto. Corre la H100.')

## 8. Conclusiones y handoff

- **Si el F1 por-clase sube** con el dataset completo: la causa raiz era el muestreo del 1 %, confirmado por el A/B (seccion 5).
- **Si NO sube significativamente** pese al dato completo: el techo es el domain-shift / temporal (24 vs 43 fechas), no el muestreo. Pivota a **US-083 (UDA fenologica: ClimID-UDA + class-aware MMD)**.
- Las ~23 clases de cola minoritarias reales seguiran dificiles aun con re-extraccion (soporte estructuralmente bajo).

> Provenance al cerrar: `US-082 @ <git_sha7> + mlflow:<run_id> + dvc:<rev>` (features `_full1438` versionadas con DVC, run del re-entreno en el server :5010).